# Valuacion PAPER: cantidad exacta y runners sin techo
## tl;dr
Los casos sinteticos conservan retornos cotizados del 200% al 100000%, sin recortar el beneficio al 200%. Una diferencia controlada muestra precio spot +100% frente a caja cotizada +10%. Tras un parcial, retorno del remanente y PnL de toda la operacion son medidas diferentes. Ningun caso acredita rentabilidad futura, fill real ni acceso actual al proveedor.

## Context & Methods
Comprobacion de calidad, grano: una generacion financiera de una posicion PAPER y una cotizacion sintetica de su cantidad restante exacta. Se ejecutan `execution/paper_cash_mark.py` y las fixtures publicas de `tests/test_paper_cash_mark.py`; no se inicia bot, API, entrenamiento, HTTP ni wallet. Fuentes complementarias: `trader/papertrading.py`, `run_bot.py`, `runtime/paper_archive.py`, `tests/test_runner_price_policy.py`.
### Key Assumptions
T0 fijo: 2026-10-08 17:00:00 UTC. Entrada: 0.1 SOL, SOL/USD sintetico de 100, coste original 10 USD, 1000 unidades raw cotizadas y referencia privada 0.02 USD. Deslizamiento adverso cero en estas fixtures para separar las identidades; la comision estimada por fill es 0.000025 SOL. Los precios de politica no son precios fisicos del token. No hay poblacion de trades reales; no se mide capacidad predictiva ni beneficio de la estrategia completa.

In [1]:
from pathlib import Path
import hashlib, sys
import pandas as pd
root = Path.cwd()
while not (root / 'execution/paper_cash_mark.py').exists():
    if root == root.parent: raise RuntimeError('Execute from the memebot3 checkout')
    root = root.parent
sys.path.insert(0, str(root))
sys.path.insert(0, str(root / 'tests'))
from test_paper_cash_mark import entry, capture, quote, OWNER, TOKEN, T0
from execution import paper_cash_mark as cash
print('contract_source_sha256=' + hashlib.sha256((root / 'execution/paper_cash_mark.py').read_bytes().replace(b'\r\n', b'\n')).hexdigest())

contract_source_sha256=0e38a231b40b9128a57c48a0b5769ce19f992c92d2da345ae5e10d8b60429094


## Data
Seis casos de subida y dos reconciliaciones deterministas. Todos usan recibos originales con mint, cantidad, coste y tiempos conocidos. Las comparaciones de caducidad conservan el reloj del recibo: no lo renuevan. Las tablas bastan para estos ocho casos exactos; no existe una serie de mercado que justifique un grafico de rendimiento.

In [2]:
cases = []
for target in (200, 500, 1000, 5000, 10000, 100000):
    mark = capture(entry(), quote(output=100000000 * (1 + target // 100)))
    values = mark.to_dict()['values']
    assert abs(values['gross_remaining_return_pct'] - target) < 1e-8
    cases.append({'Retorno sintetico (%)': target, 'Caja estimada (USD)': values['quoted_proceeds_usd'],
                  'Retorno cotizado (%)': values['gross_remaining_return_pct'], 'Cantidad raw': 1000})
display(pd.DataFrame(cases).style.format({'Caja estimada (USD)': '{:,.2f}', 'Retorno cotizado (%)': '{:,.0f}'}))

,Retorno sintetico (%),Caja estimada (USD),Retorno cotizado (%),Cantidad raw
0,200,30.00,200,1000
1,500,60.00,500,1000
2,1000,110.00,"1,000",1000
3,5000,510.00,"5,000",1000
4,10000,"1,010.00","10,000",1000
5,100000,"10,010.00","100,000",1000


## Results
La subida no se limita en el contrato de valoracion. La politica de salida protege el remanente por caida porcentual desde el pico; eso no garantiza vender al pico ni ejecutarse al precio de esta observacion.

In [3]:
spot_return = (.04 / .02 - 1) * 100
whole = capture(entry()).to_dict()['values']
part_entry = entry(qty_lamports=500, realized_qty=500, realized_proceeds_usd=7., estimated_fees_usd=.005)
part_mark = capture(part_entry, quote(500, 60000000))
part = part_mark.to_dict()['values']
assert abs(whole['gross_remaining_return_pct'] - 10) < 1e-10
assert abs(part['gross_remaining_return_pct'] - 20) < 1e-10
assert abs(part['estimated_total_liquidation_net_pnl_usd'] - 2.9925) < 1e-10
display(pd.DataFrame([{'Caso': 'Spot (proxy no vendible)', 'Retorno (%)': spot_return, 'Caja restante (USD)': None},
    {'Caso': 'Cotizacion total de 1000 raw', 'Retorno (%)': whole['gross_remaining_return_pct'], 'Caja restante (USD)': whole['quoted_proceeds_usd']},
    {'Caso': 'Remanente de 500 raw', 'Retorno (%)': part['gross_remaining_return_pct'], 'Caja restante (USD)': part['quoted_proceeds_usd']}]))
print(f"PnL neto estimado de toda la operacion parcial: {part['estimated_total_liquidation_net_pnl_usd']:.4f} USD")

,Caso,Retorno (%),Caja restante (USD)
0,Spot (proxy no vendible),100.0,NaN
1,Cotizacion total de 1000 raw,10.0,11.0
2,Remanente de 500 raw,20.0,6.0


PnL neto estimado de toda la operacion parcial: 2.9925 USD


In [4]:
import datetime as dt
mark = capture(entry())
assert cash.checked_price(mark, entry(), token=TOKEN, owner=OWNER, now=T0 + dt.timedelta(seconds=11)) is None
assert cash.checked_price(part_mark, entry(), token=TOKEN, owner=OWNER, now=T0) is None
assert mark.to_dict()['valued_at'] == T0.isoformat()
print('PASS: edad de 11 segundos y cantidad de otra generacion permanecen desconocidas; recibo sin renovar.')

PASS: edad de 11 segundos y cantidad de otra generacion permanecen desconocidas; recibo sin renovar.


## Takeaways
El monitor principal y la salida standalone PAPER consumen caja cotizada para sus decisiones, sin fallback spot cuando falta esa prueba. Los picos mixtos previos permanecen diagnosticos; nuevas valoraciones y extremos mantienen sus recibos originales en cartera y archivo. Los fills re-cotizan su propia cantidad y no se falsifican al suelo de trailing.
Pendiente: adaptar intenciones y picos por brazo en investigacion prospectiva, preservar el FX original de cada entrada/fill en todas las recuperaciones y validar una poblacion prospectiva costada de la estrategia completa. Estas pruebas no autorizan LIVE ni prometen ganancias. El runtime debe permanecer apagado.